
# Week 3 Practical — Inversion in Python

This notebook accompanies the **Geophysics 3A** Week 3 practical. It contains three problems:

1. **Linear modeling (LLSQ):** density vs. seismic velocity.
2. **Linearization & inversion:** geometric mean thermal conductivity from normative mineralogy.
3. **Newton / Gauss–Newton:** fit a nonlinear conductivity model to laboratory data.

> **Data**: Place `density_velocity.csv`, `thermo_mineralogy.csv`, and `tc_plag.csv` in this folder. If optional references are available, add `mineral_k_ref.csv`.

**Deliverable:** Submit this notebook (with outputs) and a PDF export. Keep units consistent and comment your code.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({
    'figure.figsize': (6.5,4.5),
    'axes.grid': True,
    'axes.spines.right': False,
    'axes.spines.top': False
})


In [ ]:

def llsq(A, d, Wd=None):
    '''Solve (optionally weighted) least squares: minimize ||Wd (A m - d)||_2.

    Parameters
    ----------
    A : (N,P) array
        Operator/design matrix.
    d : (N,) array
        Data vector.
    Wd : (N,) or (N,N) array, optional
        Weights (inverse standard deviation). If 1D, interpreted as diagonal.

    Returns
    -------
    m : (P,) array
        Model parameters.
    sigma_m : (P,) array
        1-sigma parameter uncertainties from covariance approximation.
    rms : float
        Root-mean-square of residuals in data space.
    r : (N,) array
        Residual vector d - A m.
    '''
    d = np.asarray(d).reshape(-1)
    A = np.asarray(A)
    if Wd is None:
        Aw, dw = A, d
        WTWin = np.eye(A.shape[0])
    else:
        W = np.diag(Wd) if np.ndim(Wd)==1 else np.asarray(Wd)
        Aw, dw = W @ A, W @ d
        WTWin = W.T @ W

    ATA = Aw.T @ Aw
    ATd = Aw.T @ dw
    m = np.linalg.solve(ATA, ATd)

    r = d - A @ m
    n, p = A.shape
    dof = max(n - p, 1)
    sigma2 = (r @ r) / dof
    Cov = sigma2 * np.linalg.inv(A.T @ WTWin @ A)
    sigma_m = np.sqrt(np.diag(Cov))
    rms = np.sqrt(np.mean(r**2))
    return m, sigma_m, rms, r



## Problem 1 — Linear modeling of density vs. seismic velocity

We model either $\rho = m_0 + m_1 V_P$ or $\log \rho = n_0 + n_1 V_P$ and compare fits. Units here are **RHO (kg/m³)** and **VP (km/s)**.


In [ ]:

# --- Load data ---
path = Path('density_velocity.csv')
if not path.exists():
    raise FileNotFoundError('Missing density_velocity.csv. Place the file in this folder and rerun.')

df = pd.read_csv(path)
# Try common column names; adjust if needed
rho_cols = [c for c in df.columns if c.lower().startswith('rho')]
vp_cols  = [c for c in df.columns if c.lower().startswith(('vp','v_p','pwave','p_wave'))]
if not rho_cols or not vp_cols:
    raise ValueError('Could not infer column names. Expected columns like RHO, VP. Inspect the CSV headers.')

rho = df[rho_cols[0]].to_numpy()
vp  = df[vp_cols[0]].to_numpy()

# --- Linear model in data space ---
A = np.column_stack([np.ones_like(vp), vp])
m, sm, rms, r = llsq(A, rho)
print(f"Linear model: rho = m0 + m1*Vp  =>  m0={m[0]:.3f}±{sm[0]:.3f}, m1={m[1]:.3f}±{sm[1]:.3f};  RMS={rms:.3f}")

# Plot
vp_grid = np.linspace(vp.min(), vp.max(), 200)
rho_fit = np.column_stack([np.ones_like(vp_grid), vp_grid]) @ m
plt.figure(); plt.scatter(vp, rho, s=6, alpha=0.3, label='data')
plt.plot(vp_grid, rho_fit, 'r', lw=2, label='LLSQ')
plt.xlabel('V_p (km/s)'); plt.ylabel('Density (kg/m^3)'); plt.legend(); plt.title('Linear fit'); plt.show()

# Residuals
plt.figure(); plt.axhline(0, color='k', lw=1)
plt.scatter(vp, r, s=6, alpha=0.3)
plt.xlabel('V_p (km/s)'); plt.ylabel('Residual (kg/m^3)'); plt.title('Residuals vs Vp'); plt.show()


In [ ]:

# --- Log-space alternative (if RHO>0) ---
if np.all(rho > 0):
    A = np.column_stack([np.ones_like(vp), vp])
    d = np.log(rho)
    m_log, sm_log, rms_log, r_log = llsq(A, d)
    n0, n1 = m_log
    print(f"Log model: log(rho)= n0 + n1*Vp  =>  n0={n0:.4f}±{sm_log[0]:.4f}, n1={n1:.4f}±{sm_log[1]:.4f}; RMS={rms_log:.4f}")

    vp_grid = np.linspace(vp.min(), vp.max(), 200)
    rho_fit = np.exp(np.column_stack([np.ones_like(vp_grid), vp_grid]) @ m_log)
    plt.figure(); plt.scatter(vp, rho, s=6, alpha=0.3, label='data')
    plt.plot(vp_grid, rho_fit, 'r', lw=2, label='LLSQ (log-space back-transform)')
    plt.xlabel('V_p (km/s)'); plt.ylabel('Density (kg/m^3)'); plt.title('Log-space linearization'); plt.legend(); plt.show()
else:
    print('Some densities are non-positive; skipping log-space fit.')



## Problem 2 — Linearization: effective conductivity from normative mineralogy

Assume a geometric mean model $k_{\text{eff}} = \prod_i k_i^{w_i}$. Taking logs yields a linear system in $m_i = \log k_i$ with matrix $A_{ji}=w_{ji}$.


In [ ]:

# --- Load dataset with k_eff and mineral fractions ---
path = Path('thermo_mineralogy.csv')
if not path.exists():
    raise FileNotFoundError('Missing thermo_mineralogy.csv. Place the file in this folder and rerun.')

df = pd.read_csv(path)
# Identify columns: expect one column for keff and several fraction columns that sum to ~1
k_cols = [c for c in df.columns if c.lower().startswith(('k_eff','keff','k'))]
frac_cols = [c for c in df.columns if c.lower().startswith(('w_','frac','norm_','min_'))]
if not k_cols or len(frac_cols) < 2:
    raise ValueError('Could not infer keff and fraction columns. Ensure one column is k_eff and ≥2 columns of mineral fractions.')

k_eff = df[k_cols[0]].to_numpy()
W = df[frac_cols].to_numpy()

# Sanity check: fractions sum
sumW = W.sum(axis=1)
print('Fraction sums: mean={:.3f}, std={:.3f}'.format(sumW.mean(), sumW.std()))

# Build linear system in log-space
if np.any(k_eff <= 0):
    raise ValueError('Non-positive k_eff encountered; cannot take log. Clean data first.')

d = np.log(k_eff)
A = W
m, sm, rms, r = llsq(A, d)

k_mineral = np.exp(m)
sigma_k = k_mineral * sm  # error propagation

# Report
mineral_names = frac_cols
for name, km, sk in zip(mineral_names, k_mineral, sigma_k):
    print(f"{name:>15s}: k = {km:.3f} ± {1.96*sk:.3f} W/m/K (approx 95% CI)")
print(f"RMS (log-space) = {rms:.4g}")



## Problem 3 — Newton / Gauss–Newton for a nonlinear conductivity model

Model:

$$k(x,T) = (m_0 + m_1 x + m_2 x^2)\,\left(\tfrac{298}{T}\right)^{m_3}.$$

Implement Gauss–Newton with optional Tikhonov parameter $\lambda$ and simple backtracking line search.


In [ ]:

# Forward and Jacobian
import numpy as np

def forward(m, xT):
    x, T = xT[:,0], xT[:,1]
    return (m[0] + m[1]*x + m[2]*x**2) * (298.0/T)**m[3]


def jacobian(m, xT):
    x, T = xT[:,0], xT[:,1]
    s = (298.0/T)**m[3]
    dk_dm0 = s
    dk_dm1 = x * s
    dk_dm2 = (x**2) * s
    dk_dm3 = (m[0] + m[1]*x + m[2]*x**2) * s * np.log(298.0/T)
    return np.column_stack([dk_dm0, dk_dm1, dk_dm2, dk_dm3])


def gauss_newton(xT, d, m0, lam=0.0, maxit=20, tol=1e-6):
    m = np.array(m0, dtype=float)
    history = []
    for it in range(1, maxit+1):
        f = forward(m, xT)
        r = d - f
        F = jacobian(m, xT)
        A = F.T @ F + (lam**2) * np.eye(F.shape[1])
        g = F.T @ r
        dm = np.linalg.solve(A, g)
        # Backtracking line search
        alpha = 1.0
        phi0 = np.linalg.norm(r)
        while True:
            m_try = m + alpha*dm
            r_try = d - forward(m_try, xT)
            if np.linalg.norm(r_try) <= (1 - 1e-4*alpha) * phi0 or alpha < 1/64:
                break
            alpha *= 0.5
        m = m_try
        rms = np.sqrt(np.mean(r_try**2))
        history.append((it, rms, alpha))
        if np.linalg.norm(alpha*dm) < tol*(np.linalg.norm(m)+tol):
            break
    # Covariance approximation
    F = jacobian(m, xT)
    r = d - forward(m, xT)
    dof = max(len(d) - len(m), 1)
    sigma2 = (r @ r) / dof
    Cov = sigma2 * np.linalg.inv(F.T @ F + (lam**2) * np.eye(F.shape[1]))
    sig_m = np.sqrt(np.diag(Cov))
    return m, sig_m, np.array(history)


In [ ]:

# --- Load lab dataset ---
from pathlib import Path
path = Path('tc_plag.csv')
if not path.exists():
    raise FileNotFoundError('Missing tc_plag.csv. Place the file in this folder and rerun.')

dat = pd.read_csv(path, header=None)
if dat.shape[1] < 3:
    dat = pd.read_csv(path)
    if not {'An','T','k'}.issubset(set(dat.columns)):
        raise ValueError('Expect 3 columns: x(An), T(K), k.')
    X = dat[['An','T']].to_numpy()
    k = dat['k'].to_numpy()
else:
    arr = dat.to_numpy()
    X = arr[:, :2]
    k = arr[:, 2]

# Starting model and run
m0 = np.array([2.0, 0.1, -0.02, 0.8])
m, sm, hist = gauss_newton(X, k, m0, lam=0.0)
print('Gauss–Newton solution:')
for i,(mi,si) in enumerate(zip(m, sm)):
    print(f" m[{i}] = {mi:.4f} ± {1.96*si:.4f} (≈95% CI)")
print('Iter  RMS   alpha')
for it, rms, a in hist:
    print(f"{it:>3d}  {rms:7.4f}  {a:4.2f}")

# Visualization: curves at a few An values
colors = ['k','r','g','b']
for xi, col in zip([0.0, 0.2, 0.55, 1.0], colors):
    TT = np.linspace(np.min(X[:,1]), np.max(X[:,1]), 200)
    kk = forward(m, np.c_[xi*np.ones_like(TT), TT])
    plt.plot(TT, kk, color=col, lw=2, label=f'An={xi}')
plt.scatter(X[:,1], k, c=X[:,0], cmap='cool', s=16, alpha=0.6, label='data')
plt.xlabel('Temperature (K)'); plt.ylabel('k (W/m/K)'); plt.legend(); plt.title('Fitted conductivity'); plt.show()



### Sensitivity experiments
- Change the starting model `m0` (make one parameter far from the solution) and observe convergence/divergence.
- Try nonzero `lam` (e.g., 0.2–0.5) to stabilize if needed; discuss the tradeoff.
